In [14]:
"""
===========================================================
SECONDARY WEIGHT ESTIMATION
Author: MONISHA VIJAYAN (AE25M037)
===========================================================
"""

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
# =========================================================
# ------------------- CONSTANTS ----------------------------
# =========================================================

rho = 1.225
g = 9.81
b = 2
S = 0.9015
AR = b**2 / S
payload_mass_drop = 1
payload_mass = 2.1
# FROM OPTIMAL VELOCITIES
V_cruise = 18.6858
V_climb = 24.4720
V_climb2= 22.5041
V_cruise_2 = 17.1832
V_des = 24.4720
V_des2 =22.5041
V_loiter = 24.5919
cruise_dist_1 = 40000
cruise_dist_2 = 40000
height_cruise = 200
gamma_climb = 8
gamma_des = 8
C_d_0 = 0.025
e = 0.8
Initial_weight = 7


# =========================================================
# ---------------- POWER FOR PHASES ---------------------------
# =========================================================

def Power_climb(V_cl, gamma_climb, Weight):
    """Climb power requirement"""
    C_l = Weight*np.cos(np.deg2rad(gamma_climb))/(0.5*rho*V_cl**2*S)
    C_d = C_d_0 + C_l**2/(np.pi*AR*e)
    D_cl = 0.5*rho*V_cl**2*S*C_d
    T_cl = D_cl + Weight*np.sin(np.deg2rad(gamma_climb))
    return T_cl * V_cl

def Power_cruise(V_cr, Weight):
    return Power_climb(V_cr, 0, Weight)

def Power_descent(V_des, gamma_des, Weight):
    return abs(Power_climb(V_des, -gamma_des, Weight))

def Power_loiter(V_loi, Weight):
    return Power_cruise(V_loi, Weight)


# =========================================================
# ----------------- TIME CALCULATIONS ---------------------
# =========================================================

t_cr_1_sec = cruise_dist_1 / V_cruise
t_cr_2_sec = cruise_dist_2 / V_cruise_2
t_loiter_sec = 15 * 60
t_cl_1_sec = height_cruise / (V_climb*np.sin(np.deg2rad(gamma_climb)))
t_cl_2_sec = height_cruise / (V_climb2*np.sin(np.deg2rad(gamma_climb)))
t_des_1_sec = height_cruise / (V_des*np.sin(np.deg2rad(gamma_des)))
t_des_2_sec = height_cruise / (V_des2*np.sin(np.deg2rad(gamma_des)))


# =========================================================
# ----------------- ENERGY FOR DIFFERENT PHASES -----------
# =========================================================

def Total_energy(Weight):
    """Total mission energy"""
    energy_climb = Power_climb(V_climb, gamma_climb, Weight) * t_cl_1_sec
    energy_cruise_1 = Power_cruise(V_cruise, Weight) * t_cr_1_sec
    energy_cruise_2 = Power_cruise(V_cruise_2, Weight-payload_mass_drop*g) * t_cr_2_sec
    energy_descent = Power_descent(V_des, gamma_des, Weight-payload_mass_drop*g) * t_des_1_sec
    energy_loiter = Power_loiter(V_loiter, Weight) * t_loiter_sec

    return energy_climb + energy_cruise_1 + energy_cruise_2 + energy_descent + energy_loiter


# =========================================================
# ----------------- LOAD EXCEL DATA -----------------------
# =========================================================
Data = pd.read_excel("Drone_dataset.xlsx")
Data = Data[Data["Empty Weight (kg)"]/Data["MTOW (kg)"] <= 0.55]
Mtow_array = Data["MTOW (kg)"].to_numpy()
Empty_weight_array = Data["Empty Weight (kg)"].to_numpy()
Empty_weight_ratios = Empty_weight_array / Mtow_array
# =========================================================
# ------------- EMPTY WEIGHT REGRESSION -------------------
# =========================================================
logx = np.log10(Mtow_array)
logy = np.log10(Empty_weight_ratios)
# LEAST SQUARE REGRESSION
A = np.column_stack((logx, np.ones_like(logx)))
Soln, _, _, _ = np.linalg.lstsq(A, logy, rcond=None)
L = Soln[0]
loga = Soln[1]
a = 10**loga
print("\nRegression results")
print("------------------")
print("a =", a)
print("L =", L)


Regression results
------------------
a = 0.5603865979148953
L = -0.07534699303142871


In [15]:
# =========================================================
# ------------- ITERATIVE WEIGHT SOLVER -------------------
# =========================================================
W0 = Initial_weight
Weight_array = []
W_b = 1.287
W_pp = 0.145 + 0.017
power_density = 300
for _ in range(40):
    Weight_array.append(W0)
    W = W0 * g
    P_climb = Power_climb(V_climb, gamma_climb, W)
    P_cruise = Power_cruise(V_cruise, W)
    P_loiter = Power_loiter(V_loiter, W)
    P_max = max(P_climb, P_cruise, P_loiter)
    # W_pp = 1200*g # P_max / power_density
    W_empty = a * W0**L#  * W0
    W0_new =  (payload_mass + W_b + W_pp)/(1-(W_empty - W_pp/W0))

    W0 = W0_new

final_weight = W0
# =========================================================
# ------------- COMPONENT BREAKDOWN -----------------------
# =========================================================
#Battery_weight = Total_energy(final_weight*g) / energy_density
Empty_weight = final_weight*a * final_weight**L
Payload = payload_mass

print("\n================ FINAL RESULTS =================")
print(f"Final MTOW        : {final_weight:.3f} kg")
print(f"Empty weight      : {Empty_weight:.3f} kg")
print(f"Payload           : {Payload:.3f} kg")
# Power reporting
print("\nPower Requirements (W)")
print("----------------------")
print("Climb   :", Power_climb(V_climb, gamma_climb, final_weight*g))
print("Cruise  :", Power_cruise(V_cruise, final_weight*g))
print("Descent :", Power_descent(V_des, gamma_des, final_weight*g))
print("Loiter  :", Power_loiter(V_loiter, final_weight*g))
print("Climb 2  :", Power_climb(V_climb2, gamma_climb, (final_weight-2)*g))
print("Cruise 2 :", Power_cruise(V_cruise_2, (final_weight-1)*g))
print("Descent 2:", Power_descent(V_des2, gamma_des,(final_weight-1)*g))

# =========================================================
# ------------- ENERGY CONSUMED DURING EACH PHASE ----------
# =========================================================

climb = Power_climb(V_climb, gamma_climb, final_weight*g) * t_cl_1_sec #/ 3600
cruise = Power_cruise(V_cruise, final_weight*g) * t_cr_1_sec #/ 3600
descent = Power_descent(V_des, gamma_des, final_weight*g) * t_des_1_sec #/ 3600
loiter = Power_loiter(V_loiter, final_weight*g) * t_loiter_sec #/ 3600

climb2 = Power_climb(V_climb2, gamma_climb, (final_weight-1)*g) * t_cl_2_sec #/ 3600
cruise2 = Power_cruise(V_cruise_2, (final_weight-1)*g) * t_cr_2_sec #/ 3600
descent2 = Power_descent(V_des2, gamma_des, (final_weight-1)*g) * t_des_2_sec #/ 3600

print("====== ENERGY CONSUMED (Joules) ======")
print("Climb :", climb)
print("Cruise :", cruise)
print("Descent :", descent)
print("Loiter :", loiter)

print("------- AFTER PAYLOAD DROP --------")
print("Climb2 :", climb2)
print("Cruise2 :", cruise2)
print("Descent2 :", descent2)

Total_energy_joules = climb + cruise + descent + loiter + climb2 + cruise2 + descent2
print("TOTAL ENERGY CONSUMED (Joules) :", Total_energy_joules)

battery_Mah = 16000
battery_energy = battery_Mah*11.2*3.6

print("Battery Energy (Joules) :", battery_energy)

# =========================================================
# -------------T DURING EACH PHASE------------
# =========================================================
print("====== THRUST REQUIRED (N) ======")

T_climb = Power_climb(V_climb, gamma_climb, final_weight*g)/V_climb
T_cruise = Power_cruise(V_cruise, final_weight*g)/V_cruise
T_descent = Power_descent(V_des, gamma_des, final_weight*g)/V_des
T_loiter = Power_loiter(V_loiter, final_weight*g)/V_loiter

print("Climb thrust :", T_climb)
print("Cruise thrust :", T_cruise)
print("Descent thrust :", T_descent)
print("Loiter thrust :", T_loiter)

print("------- AFTER PAYLOAD DROP --------")

T_climb2 = Power_climb(V_climb2, gamma_climb, (final_weight-1)*g)/V_climb2
T_cruise2 = Power_cruise(V_cruise_2, (final_weight-1)*g)/V_cruise_2
T_descent2 = Power_descent(V_des2, gamma_des, (final_weight-1)*g)/V_des2

print("Climb2 thrust :", T_climb2)
print("Cruise2 thrust :", T_cruise2)
print("Descent2 thrust :", T_descent2)
Treq=max(T_climb,T_climb2,T_cruise,T_cruise2,T_descent,T_descent2,T_loiter)

# =========================================================
# ----------------- PLOTS --------------------------------
# =========================================================
os.makedirs("figures", exist_ok=True)
# ---------------------------------------------------------
# Convergence Plot
# ---------------------------------------------------------

iters = list(range(1, len(Weight_array) + 1))




================ FINAL RESULTS =================
Final MTOW        : 6.592 kg
Empty weight      : 3.205 kg
Payload           : 2.100 kg

Power Requirements (W)
----------------------
Climb   : 449.7547134396071
Cruise  : 126.40398767720482
Descent : 9.289806655818479
Loiter  : 232.91268539018174
Climb 2  : 312.7561157135373
Cruise 2 : 98.47399121134212
Descent 2: 6.819801523603508
====== ENERGY CONSUMED (Joules) ======
Climb : 26410.76318874767
Cruise : 270588.33483651717
Descent : 545.521539462464
Loiter : 209621.41685116355
------- AFTER PAYLOAD DROP --------
Climb2 : 22376.738258812817
Cruise2 : 229233.18406662816
Descent2 : 435.49660952760775
TOTAL ENERGY CONSUMED (Joules) : 759211.4553508596
Battery Energy (Joules) : 645120.0
====== THRUST REQUIRED (N) ======
Climb thrust : 18.378339058499797
Cruise thrust : 6.764708370912929
Descent thrust : 0.3796096214374991
Loiter thrust : 9.471113878560898
------- AFTER PAYLOAD DROP --------
Climb2 thrust : 15.571200264253577
Cruise2 thrust 

In [16]:
plt.figure(figsize=(8, 5))
plt.plot(range(len(Weight_array)), Weight_array)
plt.xlabel("Iteration")
plt.ylabel("Weight (kg)")
plt.title("Weight Convergence")
plt.annotate(f'{final_weight:.3f} kg', xy=(iters[-1], final_weight), xytext=(-60, 8), textcoords='offset points',
                 arrowprops=dict(arrowstyle='->', color='tab:blue'), fontsize=10, color='tab:blue')
plt.grid(True)
plt.savefig("figures/weight_convergence.png", dpi=300)
plt.close()

In [17]:
import math
import numpy as np


# =========================================================
# Characteristic parameters (paper equations)
# =========================================================

def thrust_characteristics(beta):
    J0T = 0.4559*beta + 0.1574
    dCT = 0.0205*beta - 0.0073
    JmT = 0.2275*beta + 0.0792
    CT0 = -0.034*beta**2 + 0.13306*beta - 0.00287
    return CT0, dCT, JmT, J0T


def power_characteristics(beta):
    J0P = 0.3839*beta + 0.3553
    dCP = 0.0249*beta - 0.0117
    JmP = 0.2670*beta + 0.1192
    CP0 = 0.017716*beta**2 + 0.0064*beta + 0.014084
    return CP0, dCP, JmP, J0P


# =========================================================
# Polynomial coefficients
# =========================================================

def thrust_poly_coeffs(beta):

    CT0, dCT, JmT, J0T = thrust_characteristics(beta)

    bT2 = -dCT/(JmT**2)
    bT1 = dCT/JmT - CT0/J0T
    bT0 = CT0

    return bT2, bT1, bT0


def power_poly_coeffs(beta):

    CP0, dCP, JmP, J0P = power_characteristics(beta)

    A = np.array([
        [J0P**3, J0P**2, J0P, 1],
        [JmP**3, JmP**2, JmP, 0],
        [3*JmP**2, 2*JmP, 1, 0],
        [0, 0, 0, 1]
    ])

    B = np.array([
        0,
        dCP - CP0*JmP/J0P,
        -CP0/J0P,
        CP0
    ])

    return np.linalg.solve(A, B)


# =========================================================
# Main sizing tool
# =========================================================

def size_prop_motor(mtow_kg, T,speed, battery_voltage, beta, rho=1.225):

    g = 9.81
    W = mtow_kg
    T =T
    V = V_climb

    # Optimal advance ratio (paper Eq. 22)
    Jopt = 0.2684*beta + 0.0879

    # Polynomial coefficients
    bT2, bT1, bT0 = thrust_poly_coeffs(beta)
    bP3, bP2, bP1, bP0 = power_poly_coeffs(beta)

    # Evaluate coefficients at Jopt
    CT = bT2*Jopt**2 + bT1*Jopt + bT0
    CP = bP3*Jopt**3 + bP2*Jopt**2 + bP1*Jopt + bP0

    # Diameter (paper Eq. 25)
    D = math.sqrt(T * Jopt**2 / (CT * rho * V**2))

    # RPM
    n = V/(Jopt*D)
    rpm = 60*n

    # Power (paper Eq. 23)
    P = CP * rho * n**3 * D**5

    Kv = rpm/(0.8*battery_voltage)

    print("\n===== PAPER-CORRECT RESULTS =====")
    print(f"CT(Jopt) = {CT:.4f}")
    print(f"CP(Jopt) = {CP:.4f}")
    print(f"Diameter = {D*39.37:.1f} in")
    print(f"RPM      = {rpm:.0f}")
    print(f"Power    = {P:.0f} W")
    print(f"Kv       = {Kv:.0f}")
    print("================================")

    return D, rpm, P

if __name__ == "__main__":

    size_prop_motor(
        mtow_kg= W0,
        T = Treq,
        speed=V_climb,
        battery_voltage= 11.4,
        beta=1.6
    )


===== PAPER-CORRECT RESULTS =====
CT(Jopt) = 0.0463
CP(Jopt) = 0.0605
Diameter = 15.0 in
RPM      = 7455
Power    = 1137 W
Kv       = 817
